##### 🌾 CSIRO Biomass Prediction - Two-Stream ConvNeXt Inference

## Reference:https://www.kaggle.com/code/none00000/lb-0-57-infer-model-code

## 📊 Performance
- **Public LB Score:** 0.58
- **Local CV Score:** 0.XXX ± 0.XXX   ※wait for training notebook!
- **Model:** ConvNeXt-Tiny with Two-Stream Architecture
- **Ensemble:** 5-Fold + 3-View TTA

---

## 🎯 What's in This Notebook?

This inference notebook implements a **two-stream architecture** for predicting pasture biomass from top-view images. Key features:

- ✅ **Two-Stream CNN**: Processes left/right image halves independently
- ✅ **Three-Head Regression**: Dedicated heads for Total, GDM, and Green biomass
- ✅ **5-Fold Ensemble**: Robust predictions through cross-validation
- ✅ **Test-Time Augmentation**: Original + HFlip + VFlip (3 views)
- ✅ **Clean & Documented Code**: Easy to understand and modify

---

## 💬 Feedback & Discussion

If you find this notebook helpful:
- 👍 **Please upvote** to support my work
- 💬 **Leave a comment** with your questions or suggestions
- 🔔 **Follow me** for more competitions and insights

**Questions? Issues?** Feel free to comment below, and I'll respond ASAP!



In [4]:
"""
CSIRO Biomass Competition - Inference Pipeline (TTA + Ensemble)
================================================================================
This script performs predictions on test data using trained models.

Pipeline Overview:
1. Test Data Preparation: Load CSV → Extract unique images
2. Model Loading: Load 5-Fold trained models
3. TTA Inference: 3 Views × 5-Fold Ensemble
4. Post-processing: 3 predictions → Reconstruct 5 targets
5. Submission Creation: Wide format → Long format conversion
"""

from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional
from collections import OrderedDict
import os
import gc

from catboost import CatBoostRegressor
import joblib
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
import cv2
from tqdm import tqdm


# ============================================================================
# Configuration Management
# ============================================================================

@dataclass
class InferenceConfig:
    """
    Data class for managing inference pipeline configuration.
    
    Items that must match training configuration:
    - model_name
    - img_size
    - target column names
    """
    
    # Path settings
    base_path: Path = Path('csiro-biomass')
    test_csv: Path = field(init=False)
    test_image_dir: Path = field(init=False)
    model_dir: Path = Path('out')
    backbone_dir: Path = Path('vit_large_patch16_dinov3.pth')
    submission_file: str = 'submission.csv'
    ridge_dir: Path = Path('ridge_models.pkl')
    catboost_dir : Path = Path('cached_folds_date_state_20')
    knn_fold_dir : Path = Path('cached_folds_date_state_20')
    # Model settings (must match training)
    model_name: str = 'vit_large_patch16_dinov3'
    img_size: int = 768
    
    # Device settings
    device: torch.device = field(default_factory=lambda: torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
    ))
    
    # Inference settings
    batch_size: int = 8
    num_workers: int = 1
    n_folds: int = 5
    
    # Target settings (must match training)
    train_target_cols: list[str] = field(default_factory=lambda: [
        'Dry_Total_g', 'GDM_g', 'Dry_Green_g'
    ])
    
    all_target_cols: list[str] = field(default_factory=lambda: [
        'Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g'
    ])
    
    def __post_init__(self) -> None:
        """Construct paths after initialization"""
        self.test_csv = self.base_path / 'test.csv'
        self.test_image_dir = self.base_path / 'test'
    
    def display_info(self) -> None:
        """Display configuration information"""
        print(f"{'='*70}")
        print(f"Inference Configuration")
        print(f"{'='*70}")
        print(f"Device: {self.device}")
        print(f"Backbone: {self.model_name}")
        print(f"Image Size: {self.img_size}x{self.img_size}")
        print(f"Batch Size: {self.batch_size}")
        print(f"Ensemble: {self.n_folds}-Fold")
        print(f"TTA: 3 Views (Original, HFlip, VFlip)")
        print(f"{'='*70}\n")


# ============================================================================
# TTA Augmentation
# ============================================================================

class TTATransformFactory:
    """
    Factory class for generating Test Time Augmentation transforms.
    
    Provides 3 different views:
    1. Original (no augmentation)
    2. Horizontal flip
    3. Vertical flip
    """
    
    def __init__(self, img_size: int):
        """
        Args:
            img_size: Image size after resizing
        """
        self.img_size = img_size
        
        # Base transforms common to all views
        self.base_transforms = [
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ]
    
    def get_tta_transforms(self) -> list[A.Compose]:
        """
        Generate 3 transform pipelines for TTA.
        
        Returns:
            List of 3 Albumentations Compose objects
            
        Why not: Not adding more TTA variations
            → Considering trade-off with inference time
        """
        # View 1: Original
        original = A.Compose([
            A.Resize(self.img_size, self.img_size),
            *self.base_transforms
        ])
        
        # View 2: Horizontal flip
        hflip = A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Resize(self.img_size, self.img_size),
            *self.base_transforms
        ])
        
        # View 3: Vertical flip
        vflip = A.Compose([
            A.VerticalFlip(p=1.0),
            A.Resize(self.img_size, self.img_size),
            *self.base_transforms
        ])

        rot180 = A.Compose([
            A.HorizontalFlip(p=1.0),
            A.VerticalFlip(p=1.0),
            A.Resize(self.img_size, self.img_size),
            *self.base_transforms
        ])

        transp = A.Compose([
            A.Transpose(p=1.0),
            A.Resize(self.img_size, self.img_size),
            *self.base_transforms
        ])
        
        return [original, hflip, vflip, rot180, transp]


# ============================================================================
# Dataset
# ============================================================================

class TestBiomassDataset(Dataset):
    """
    Two-stream dataset for testing.
    
    Accepts a specific transform pipeline for TTA and applies
    the same augmentation to both left and right images.
    
    Returns:
        tuple: (img_left, img_right)
    """
    
    def __init__(
        self,
        df: pd.DataFrame,
        transform_pipeline: A.Compose,
        image_dir: Path
    ):
        """
        Args:
            df: DataFrame containing image paths
            transform_pipeline: Augmentation pipeline to apply
            image_dir: Image directory path
        """
        self.df = df
        self.transform = transform_pipeline
        self.image_dir = image_dir
        self.image_paths = df['image_path'].values
    
    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Get one sample.
        
        Args:
            idx: Sample index
            
        Returns:
            (left_image, right_image)
            
        Why not: Not applying different augmentations to left/right as in training
            → During TTA, apply same transform to both images to preserve symmetry
        """
        img_path = self.image_paths[idx]
        full_path = self.image_dir / Path(img_path).name
        
        # Load image (return black image on error)
        image = cv2.imread(str(full_path))
        
        if image is None:
            print(f"Warning: Failed to load image: {full_path} → Returning black image")
            image = np.zeros((1000, 2000, 3), dtype=np.uint8)
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Split into left and right
        height, width = image.shape[:2]
        mid_point = width // 2
        img_left = image[:, :mid_point]
        img_right = image[:, mid_point:]
        
        # Apply same transform to both
        img_left_tensor = self.transform(image=img_left)['image']
        img_right_tensor = self.transform(image=img_right)['image']
        
        return img_left_tensor, img_right_tensor


# ============================================================================
# Model
# ============================================================================
class BackboneModel(nn.Module):
    """
    Two-stream, three-head regression model.
    
    Exactly the same architecture as during training.
    """
    
    def __init__(self, model_name: str, pretrained: bool = False, is_linear=False):
        """
        Args:
            model_name: timm model name
            pretrained: Always False (weights loaded separately)
        """
        super().__init__()
        self.is_linear=is_linear
        # Shared backbone
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
        )
        nf = self.backbone.num_features
        image_feature_dim = nf * 2
    
    def forward(
        self,
        img_left: torch.Tensor,
        img_right: torch.Tensor,
        aux_cont=None
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Forward pass.
        
        Args:
            img_left: Left image [B, C, H, W]
            img_right: Right image [B, C, H, W]
            
        Returns:
            (total_pred, gdm_pred, green_pred) each [B, 1]
        """
        fl = self.backbone(img_left)
        fr = self.backbone(img_right)
        image_features = torch.cat([fl, fr], dim=1)
        return image_features

class BiomassSimpleMLP(nn.Module):
    def __init__(self, image_feature_dim):
        super().__init__()

        self.head_total = self._create_head(image_feature_dim)
        self.head_gdm = self._create_head(image_feature_dim)
        self.head_clover = self._create_head(image_feature_dim)
        self.head_green = self._create_head(image_feature_dim)

    def _create_head(self, feature_dim):
        return nn.Sequential(
        nn.Linear(feature_dim, feature_dim//2), 
        nn.GELU(),
        nn.Dropout(0.3),
        nn.Linear(feature_dim//2, feature_dim//4),
        nn.GELU(),
        nn.Dropout(0.3),
        nn.Linear(feature_dim//4, 1)
        )
    def forward(self, feats):
        p_total = F.softplus(self.head_total(feats))
        p_gdm = F.softplus(self.head_gdm(feats))
        p_clover = F.softplus(self.head_clover(feats))
        p_green = F.softplus(self.head_green(feats))
        p_dead = p_total - p_gdm

        return (p_total, p_gdm, p_green, p_clover, p_dead)
# ============================================================================
# Model Loader
# ============================================================================

class ModelLoader:
    """
    Class for loading trained models.
    
    Handles weights saved with DataParallel.
    """
    
    def __init__(self, config: InferenceConfig):
        """
        Args:
            config: Configuration object
        """
        self.config = config
        
    def load_model(self, path) -> nn.Module:
        print(f"\nLoading {path} model")
        
        model_path = path
        
        if not model_path.exists():
            raise FileNotFoundError(f"Model file not found: {model_path}")
        
        # Initialize model
        backbone_model = BackboneModel(self.config.model_name, pretrained=False)
        
        # Load weights
        state_dict = torch.load(model_path, map_location=self.config.device)
        
        # Handle DataParallel (remove 'module.' prefix)
        state_dict = self._remove_dataparallel_prefix(state_dict)
        
        backbone_model.backbone.load_state_dict(state_dict)
        backbone_model.eval()  # Evaluation mode
        backbone_model.to(self.config.device)

        if torch.cuda.device_count() > 1:
            print(f"   -> ⚡ Using {torch.cuda.device_count()} GPUs for Backbone")
            backbone_model.to('cuda:0') 
            backbone_model = nn.DataParallel(backbone_model)
        
        print(f"✓ Successfully loaded {path} model\n")
        return backbone_model
    
    def load_ridge_models(self):
        ridge_models = joblib.load(self.config.ridge_dir)
        return ridge_models

    def load_knn_models(self):
        """
        Loads the 5-Fold Multi-Output KNN models.
        """
        knn_models = []
        print(f"Loading KNN models from {self.config.knn_fold_dir}...")

        for fold in range(self.config.n_folds):
            # Matches the filename from train_knn: "knn_model_fold0.pkl"
            # Adjust 'fold+1' or 'fold' depending on your folder naming convention
            # (Your previous code used 0-based indexing for filenames usually)
            path = os.path.join(
                self.config.knn_fold_dir, 
                f"knn_model_fold{fold}.pkl"
            )
            
            if os.path.exists(path):
                model = joblib.load(path)
                knn_models.append(model)
                print(f"  ✅ Loaded KNN Fold {fold}")
            else:
                print(f"  ❌ Missing KNN model: {path}")
        return knn_models

    def load_catboost_models(self):
        """
        Loads 5 CatBoost models (folds) for each target.
        Structure: {'Total': [Model_F0, Model_F1...], 'Green': [...]}
        """
        models = {}
        targets = ['Total', 'Green', 'Clover', 'GDM']
        n_folds = self.config.n_folds
        
        print(f"📂 Loading CatBoost models from: {self.config.catboost_dir}")

        for target in targets:
            models[target] = []
            
            for fold in range(n_folds):
                # e.g., 'catboost_total_fold0.cbm'
                filename = f"catboost_{target.lower()}_fold{fold}.cbm"
                path = os.path.join(self.config.catboost_dir, filename)
                
                if os.path.exists(path):
                    clf = CatBoostRegressor()
                    clf.load_model(path)
                    models[target].append(clf)
                else:
                    print(f"  ❌ Missing: {filename}")
            
            print(f"  ✅ {target}: Loaded {len(models[target])} folds")

        return models

    def load_fold_models(self) -> list[nn.Module]:
        """
        Load all 5-Fold trained models.
        
        Returns:
            List of models (each in eval mode on GPU)
            
        Raises:
            FileNotFoundError: If model file not found
        """
        print(f"\nLoading {self.config.n_folds} trained models...")
        
        models = []
        
        for fold in range(self.config.n_folds):
            model_path = self.config.model_dir / f'best_model_fold{fold}.pth'
            
            if not model_path.exists():
                raise FileNotFoundError(f"Model file not found: {model_path}")
            
            # Initialize model
            model = BiomassSimpleMLP(2048)
            
            # Load weights
            state_dict = torch.load(model_path, map_location=self.config.device)
            
            # Handle DataParallel (remove 'module.' prefix)
            state_dict = self._remove_dataparallel_prefix(state_dict)
            
            model.load_state_dict(state_dict)
            model.eval()  # Evaluation mode
            model.to(self.config.device)
            if torch.cuda.device_count() > 1:
                model.to('cuda:0')
                model = nn.DataParallel(model)
            models.append(model)
            print(f"  ✓ Fold {fold} model loaded")
        
        print(f"✓ Successfully loaded {len(models)} models\n")
        return models
    
    @staticmethod
    def _remove_dataparallel_prefix(state_dict: dict) -> dict:
        """
        Remove 'module.' prefix from DataParallel-saved weights.
        
        Args:
            state_dict: Model weight dictionary
            
        Returns:
            Weight dictionary with prefix removed
            
        Why not: Not using try-except with direct load_state_dict
            → Explicitly handling prefix presence improves readability
        """
        if not any(k.startswith('module.') for k in state_dict.keys()):
            return state_dict  # No prefix
        
        new_state_dict = OrderedDict()
        for key, value in state_dict.items():
            new_key = key.replace('module.', '')
            new_state_dict[new_key] = value
        
        return new_state_dict


# ============================================================================
# Inference Engines
# ============================================================================

class GlobalFeatureExtractor:
    def __init__(self, backbone, config):
        self.backbone = backbone
        self.config = config
        self.device = config.device
        self.backbone.to(self.device)
        self.backbone.eval()

    def get_all_embeddings(self, test_df, tta_transforms):
        """
        Runs backbone on ALL TTA variants.
        
        Returns:
            X_massive (np.ndarray): Shape (N_images * K_transforms, 2048)
            N (int): Number of original images
            K (int): Number of transforms
        """
        N = len(test_df)
        K = len(tta_transforms)
        all_folds_features = []

        print(f"--- 🚀 Starting Global Feature Extraction ({K} views) ---")
        
        # 1. Loop through each TTA Transform
        for i, transform in enumerate(tta_transforms):
            print(f"  > Processing View {i+1}/{K}...")
            
            # Setup Loader for this specific view
            dataset = TestBiomassDataset(
                test_df,
                transform,
                self.config.test_image_dir
            )
            
            loader = DataLoader(
                dataset,
                batch_size=self.config.batch_size,
                shuffle=False,
                num_workers=self.config.num_workers,
                pin_memory=True
            )
            
            # Extract features for this view
            view_feats = self._extract_one_view(loader)
            all_folds_features.append(view_feats)

        # 2. Stack Vertically
        # Result: [View1_Img1, View1_Img2... | View2_Img1, View2_Img2...]
        X_massive = np.vstack(all_folds_features)
        
        expected_rows = N * K
        assert X_massive.shape[0] == expected_rows, f"Shape mismatch! Got {X_massive.shape}, expected ({expected_rows}, ...)"
        
        print(f"--- ✅ Extraction Complete. Matrix Shape: {X_massive.shape} ---")
        return X_massive, N, K

    def _extract_one_view(self, loader):
        """Helper to run backbone on one loader"""
        feats_list = []
        with torch.inference_mode():
            for img_left, img_right in tqdm(loader, leave=False):
                img_left = img_left.to(self.device)
                img_right = img_right.to(self.device)
                
                with torch.amp.autocast('cuda'):
                    # Ensure your backbone returns (Batch, 2048)
                    batch_feats = self.backbone(img_left, img_right)
                
                feats_list.append(batch_feats.cpu().numpy())
        return np.vstack(feats_list)

class EnsemblePredictor:
    def __init__(self, mlp_models, knn_models, cat_models, config):
        self.mlp_models = mlp_models
        self.knn_models = knn_models # Dict from joblib
        self.cat_models = cat_models
        self.config = config
        self.device = self.config.device
        
        # Prep MLPs
        for m in self.mlp_models:
            m.to(self.device)
            m.eval()

    def predict_all(self, X_massive, N, K):
        """
        Predicts on the massive matrix and averages predictions back to N images.
        """
        print("--- 🧠 Running Ensemble Predictions ---")
        
        # --- 1. MLP PREDICTIONS ---
        print("  > MLP Heads...")
        mlp_preds_flat = self._predict_mlp(X_massive)
        mlp_final = self._reshape_and_avg(mlp_preds_flat, N, K)
        
        # --- 2. RIDGE/KRR PREDICTIONS ---
        print("  > Kernel Ridge...")
        ridge_preds_flat = self._predict_ridge(X_massive)
        ridge_final = self._reshape_and_avg(ridge_preds_flat, N, K)
        
        # --- 3. CATBOOST PREDICTIONS ---
        print("  > CatBoost...")
        cat_preds_flat = self._predict_cat(X_massive)
        cat_final = self._reshape_and_avg(cat_preds_flat, N, K)
        
        return mlp_final, ridge_final, cat_final

    def _predict_mlp(self, X):
        """Runs PyTorch MLP heads on numpy features"""
        # Convert entire matrix to tensor (GPU memory permitting)
        # If OOM occurs here, do this in chunks/batches
        X_tensor = torch.from_numpy(X).float().to(self.device)
        
        accum = {'total': 0, 'green': 0, 'gdm': 0, 'clover': 0, 'dead': 0}
        
        with torch.inference_mode():
            for model in self.mlp_models:
                p_total, p_gdm, p_green, p_clover, p_dead = model(X_tensor)
                
                accum['total'] += p_total.cpu().numpy().flatten()
                accum['green'] += p_green.cpu().numpy().flatten()
                accum['gdm']   += p_gdm.cpu().numpy().flatten()
                accum['clover']+= p_clover.cpu().numpy().flatten()
                accum['dead']+= p_dead.cpu().numpy().flatten()
        
        # Average over 5 folds
        n_folds = len(self.mlp_models)
        return {k: v / n_folds for k, v in accum.items()}

    def _predict_knn(self, X):
        """
        Runs inference using 5-fold KNN models.
        Unlike CatBoost, these KNNs are Multi-Output (predict all 4 targets at once).
        """
        # 1. Collect predictions from all folds
        # Each fold outputs shape (Batch_Size, 4)
        fold_preds = []
        
        for model in self.knn_models:
            # KNN predict is fast on batches
            p = model.predict(X)
            fold_preds.append(p)
        
        # 2. Average across folds (Ensemble)
        # Shape becomes (Batch_Size, 4)
        avg_preds = np.mean(fold_preds, axis=0)
        
        # 3. Unpack into dictionary
        # Mapping based on training indices [Green, Clover, GDM, Total]
        preds = {}
        preds['green']  = avg_preds[:, 0]
        preds['clover'] = avg_preds[:, 1]
        preds['gdm']    = avg_preds[:, 2]
        preds['total']  = avg_preds[:, 3]
        
        # 4. Derive Dead Mass
        # Dead = Total - (Green + Clover + GDM)
        preds['dead'] = preds['total'] - preds['gdm']
        
        return preds

    def _predict_ridge(self, X):
        """Runs Sklearn models"""
        preds = {}
        # 1. Run the existing models
        # (Assuming X is the raw 2048 features)
        preds['total']  = self.ridge_models['Total'].predict(X)
        preds['green']  = self.ridge_models['Green'].predict(X)
        preds['clover'] = self.ridge_models['Clover'].predict(X)
        preds['gdm']    = self.ridge_models['GDM'].predict(X)
        
        # 2. Derive 'Dead' (The Missing Piece)
        # Using the standard subtraction logic to ensure mass balance:
        # Dead = Total - (Everything Else)
        preds['dead'] = preds['total'] - preds['gdm']
        
        return preds

    def _predict_cat(self, X):
        """
        Runs inference using 5-fold CatBoost models.
        """
        preds = {}
        targets = ['Total', 'Green', 'Clover', 'GDM']
        
        for t in targets:
            fold_preds = []
            
            for model in self.cat_models[t]:
                # Predict on the batch
                p = model.predict(X)
                fold_preds.append(p)
                
            # Stack them to shape (5, BatchSize) then mean axis 0
            preds[t.lower()] = np.mean(fold_preds, axis=0)
        
        preds['dead'] = preds['total'] - preds['gdm']
        
        return preds

    def _reshape_and_avg(self, preds_dict, N, K):
        """
        Collapses the TTA dimension.
        Input: Dict of arrays (N*K,)
        Output: Dict of arrays (N,)
        """
        final = {}
        for key, arr in preds_dict.items():
            # 1. Split into K chunks of size N
            chunks = np.split(arr, K)
            # 2. Stack and Mean
            stacked = np.column_stack(chunks)
            final[key] = np.mean(stacked, axis=1)
            
            # Physics Check
            # final[key] = np.maximum(final[key], 0)
            
        return final

# ============================================================================
# Submission Creation
# ============================================================================

class SubmissionCreator:
    """
    Class for creating Kaggle submission CSV from predictions.
    """
    
    def __init__(self, config: InferenceConfig):
        """
        Args:
            config: Configuration object
        """
        self.config = config
    
    def create(
        self,
        predictions: dict[str, np.ndarray],
        test_df_long: pd.DataFrame,
        test_df_unique: pd.DataFrame
    ) -> None:
        """
        Create and save submission CSV from predictions.
        
        Args:
            predictions: {'total': [N], 'gdm': [N], 'green': [N]}
            test_df_long: Original test.csv (long format)
            test_df_unique: DataFrame of unique images
            
        Processing flow:
        1. Calculate 5 targets from 3 predictions
        2. Create wide format DataFrame
        3. Convert to long format (melt)
        4. Merge with sample_id
        5. Save CSV
        """
        print("Creating submission CSV...")
        
        # 1. Get 3 predictions
        pred_total = predictions['total']
        pred_gdm = predictions['gdm']
        pred_green = predictions['green']
        pred_clover = predictions['clover']
        pred_dead = predictions['dead']
        
        # 3. Create wide format DataFrame
        preds_wide = pd.DataFrame({
            'image_path': test_df_unique['image_path'],
            'Dry_Green_g': pred_green,
            'Dry_Dead_g': pred_dead,
            'Dry_Clover_g': pred_clover,
            'GDM_g': pred_gdm,
            'Dry_Total_g': pred_total
        })
        
        # 4. Convert to long format (unpivot)
        preds_long = preds_wide.melt(
            id_vars=['image_path'],
            value_vars=self.config.all_target_cols,
            var_name='target_name',
            value_name='target'
        )
        
        # 5. Merge with original test.csv to get sample_id
        submission = pd.merge(
            test_df_long[['sample_id', 'image_path', 'target_name']],
            preds_long,
            on=['image_path', 'target_name'],
            how='left'
        )
        
        # 6. Format and save
        submission = submission[['sample_id', 'target']]
        submission.to_csv(self.config.submission_file, index=False)
        
        print(f"\n🎉 Submission saved: {self.config.submission_file}")
        print("\n--- First 5 rows ---")
        print(submission.head())
        print("\n--- Last 5 rows ---")
        print(submission.tail())


# ============================================================================
# Inference Pipeline
# ============================================================================
class InferencePipeline:
    """
    Class that orchestrates the entire inference pipeline.
    """
    
    def __init__(self, config: InferenceConfig):
        """
        Args:
            config: Configuration object
        """
        self.config = config
        self.model_loader = ModelLoader(config)
        self.tta_factory = TTATransformFactory(config.img_size)
        self.submission_creator = SubmissionCreator(config)
    
    def run(self) -> None:
        """
        Execute the entire inference pipeline.
        
        Processing flow:
        1. Load test data
        2. Load models (5-Fold)
        3. TTA inference (3 Views × 5 Folds)
        4. Create submission
        """
        print(f"\n{'='*70}")
        print(f"🚀 Starting Inference Pipeline")
        print(f"{'='*70}")
        
        try:
            # 1. Load test data
            test_df_long, test_df_unique = self._load_test_data()
            backbone = self.model_loader.load_model(self.config.backbone_dir)
            feat_extractor = GlobalFeatureExtractor(backbone, self.config)
            del backbone
            torch.cuda.empty_cache()
            # 2. Load models
            mlp_models = self.model_loader.load_fold_models()
            # ridge_models = self.model_loader.load_ridge_models()
            catboost_models = self.model_loader.load_catboost_models()
            knn_models = self.model_loader.load_knn_models()
            
            # 3. TTA inference
            tta_transforms = self.tta_factory.get_tta_transforms()

            
            X_massive, N, K = feat_extractor.get_all_embeddings(test_df_unique, tta_transforms)

            predictor = EnsemblePredictor(mlp_models, knn_models, catboost_models, self.config)

            preds_mlp = predictor._predict_mlp(X_massive)
            preds_mlp_avg = predictor._reshape_and_avg(preds_mlp, N, K)

            preds_knn = predictor._predict_knn(X_massive)
            preds_knn_avg = predictor._reshape_and_avg(preds_knn, N, K)
            
            preds_catboost = predictor._predict_cat(X_massive)
            preds_catboost_avg = predictor._reshape_and_avg(preds_catboost, N, K)

            w_mlp = 0.7
            w_cat = 0.2
            w_knn = 0.1

            # 2. Loop through keys and average the arrays
            final_preds = {}
            # Ensure both dicts have the same keys ('total', 'green', 'clover', 'gdm', 'dead')
            target_keys = preds_mlp_avg.keys() 

            for key in target_keys:
                # Get the (N,) array from MLP and Ridge
                p_mlp = preds_mlp_avg[key]
                p_cat = preds_catboost_avg[key]
                p_knn = preds_knn_avg[key]
                
                # Weighted Average (Allowing negatives to cancel out!)
                avg_pred = (p_mlp * w_mlp) + (p_cat * w_cat) + (p_knn * w_knn)
                
                # 3. FINAL CLAMP (Physics Check)
                # This is the ONLY place you clamp.
                final_preds[key] = np.maximum(avg_pred, 0)

            # 4. Create submission
            self.submission_creator.create(
                final_preds,
                test_df_long,
                test_df_unique
            )
            
            print("\n✨ Inference Pipeline Completed ✨")
            
        except Exception as e:
            print(f"\n❌ Error occurred: {e}")
            raise
        
        finally:
            # Free memory
            gc.collect()
            torch.cuda.empty_cache()
    
    def _load_test_data(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        """
        Load test data.
        
        Returns:
            (test_df_long, test_df_unique)
            - test_df_long: Original long format (with sample_id)
            - test_df_unique: Unique images only
            
        Raises:
            FileNotFoundError: If test.csv not found
        """
        print(f"\nLoading test data: {self.config.test_csv}")
        
        if not self.config.test_csv.exists():
            raise FileNotFoundError(f"test.csv not found: {self.config.test_csv}")
        
        test_df_long = pd.read_csv(self.config.test_csv)
        test_df_unique = test_df_long.drop_duplicates(
            subset=['image_path']
        ).reset_index(drop=True)
        
        print(f"  Long format: {len(test_df_long)} rows")
        print(f"  Unique images: {len(test_df_unique)} images\n")
        
        return test_df_long, test_df_unique


# ============================================================================
# Main Execution
# ============================================================================

if __name__ == '__main__':
    # Initialize configuration
    config = InferenceConfig()
    config.display_info()
    
    # Run pipeline
    pipeline = InferencePipeline(config)
    pipeline.run()
    
    print("\n" + "="*70)
    print("🎊 Inference Pipeline Completed!")
    print("="*70)

Inference Configuration
Device: cuda
Backbone: vit_large_patch16_dinov3
Image Size: 768x768
Batch Size: 8
Ensemble: 5-Fold
TTA: 3 Views (Original, HFlip, VFlip)


🚀 Starting Inference Pipeline

Loading test data: csiro-biomass/test.csv
  Long format: 5 rows
  Unique images: 1 images


Loading vit_large_patch16_dinov3.pth model
✓ Successfully loaded vit_large_patch16_dinov3.pth model


Loading 5 trained models...
  ✓ Fold 0 model loaded
  ✓ Fold 1 model loaded
  ✓ Fold 2 model loaded
  ✓ Fold 3 model loaded
  ✓ Fold 4 model loaded
✓ Successfully loaded 5 models

📂 Loading CatBoost models from: cached_folds_date_state_20
  ✅ Total: Loaded 5 folds
  ✅ Green: Loaded 5 folds
  ✅ Clover: Loaded 5 folds
  ✅ GDM: Loaded 5 folds
Loading KNN models from cached_folds_date_state_20...
  ✅ Loaded KNN Fold 0
  ✅ Loaded KNN Fold 1
  ✅ Loaded KNN Fold 2
  ✅ Loaded KNN Fold 3
  ✅ Loaded KNN Fold 4
--- 🚀 Starting Global Feature Extraction (5 views) ---
  > Processing View 1/5...


  > Processing View 2/5...


  > Processing View 3/5...


  > Processing View 4/5...


  > Processing View 5/5...


--- ✅ Extraction Complete. Matrix Shape: (5, 2048) ---
Creating submission CSV...

🎉 Submission saved: submission.csv

--- First 5 rows ---
                    sample_id     target
0  ID1001187975__Dry_Clover_g   0.290568
1    ID1001187975__Dry_Dead_g  35.120906
2   ID1001187975__Dry_Green_g  30.364160
3   ID1001187975__Dry_Total_g  66.234757
4         ID1001187975__GDM_g  31.113849

--- Last 5 rows ---
                    sample_id     target
0  ID1001187975__Dry_Clover_g   0.290568
1    ID1001187975__Dry_Dead_g  35.120906
2   ID1001187975__Dry_Green_g  30.364160
3   ID1001187975__Dry_Total_g  66.234757
4         ID1001187975__GDM_g  31.113849

✨ Inference Pipeline Completed ✨

🎊 Inference Pipeline Completed!


In [ ]:
0  ID1001187975__Dry_Clover_g   0.167680
1    ID1001187975__Dry_Dead_g  37.301017
2   ID1001187975__Dry_Green_g  29.283626
3   ID1001187975__Dry_Total_g  68.008954
4         ID1001187975__GDM_g  30.707930